In [1]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8
RECURSION_DEPTHS = list(range(10, MAX_RECURSION_DEPTH + 1, 10))  # [10, 20, 30, 40, 50]

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

# ✅ New: Recursive Depth Stabilization (RDS)
@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))  # ✅ Apply stabilization

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)  # ✅ Ensure integer depth for loop

# --- Sharding Setup (Fixed for TPU v5e-1, which has only 1 device)
devices = jax.devices()  # Automatically detect TPU devices
sharding = PositionalSharding(devices)  # Adjust sharding dynamically

batch_size = 50_000
data_size = 50_000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(
        lambda xi: dppu_with_dynamic_pi_phi(xi, depth=10, scale_factor=0.5), in_axes=0
    )(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ New: Adaptive Depth Execution to Prevent Instability
def split_into_stable_depths(x, total_depth):
    """Runs dppu_with_dynamic_pi_phi in Depth=10 chunks instead of deeper recursion."""
    iterations = total_depth // 10
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=10)  # ✅ Always process in stable Depth 10 chunks
    return x

# --- Run with Stabilized Depth
for depth in RECURSION_DEPTHS:
    if depth == 10:
        output_batch = batched_dppu_processing(batch_input)
    else:
        output_batch = split_into_stable_depths(batch_input, depth)

    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

NUM_TRIALS = 10
INPUT_SIZE = 50_000

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)

for depth in RECURSION_DEPTHS:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        if depth == 10:
            result = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)
        else:
            result = split_into_stable_depths(jnp.ones((INPUT_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Size={INPUT_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

jax.devices()

# --- Investigate TPU Compilation Stability
compiled_fn_10 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((50_000,)), depth=10)
compiled_fn_50 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((50_000,)), depth=50)

print("\n🚀 XLA Compilation for Depth=10:")
print(compiled_fn_10.as_text())

print("\n🚀 XLA Compilation for Depth=50:")
print(compiled_fn_50.as_text())





Batch Output Shape (Depth=10): (50000,)
Batch Output Shape (Depth=20): (50000,)
Batch Output Shape (Depth=30): (50000,)
Batch Output Shape (Depth=40): (50000,)
Batch Output Shape (Depth=50): (50000,)

🔥 TPU Benchmark (Depth=10, Size=50000)
Avg: 0.001685, Min: 0.001252, Max: 0.003714

🔥 TPU Benchmark (Depth=20, Size=50000)
Avg: 0.001338, Min: 0.001267, Max: 0.001641

🔥 TPU Benchmark (Depth=30, Size=50000)
Avg: 0.001384, Min: 0.001246, Max: 0.001549

🔥 TPU Benchmark (Depth=40, Size=50000)
Avg: 0.001465, Min: 0.001289, Max: 0.001805

🔥 TPU Benchmark (Depth=50, Size=50000)
Avg: 0.001479, Min: 0.001359, Max: 0.001540

🚀 XLA Compilation for Depth=10:
module @jit_dppu_with_dynamic_pi_phi attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<50000xf32> {mhlo.layout_mode = "default"}, %arg1: tensor<i32> {mhlo.layout_mode = "default"}) -> (tensor<50000xf32> {jax.result_info = "", mhlo.layout_mode = "default"}) {
    %0 = call @dppu_with_

/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
